# NHANES 2013-2014: building the dataset for the diabetes risk classifier

**Part A** (run once): downloads the files, merges them, filters the sample, builds the label and saves two files (`nhanes_2013_merged.csv`, `codebook.csv`).

**Part B** (can be rerun as often as needed): starts from the CSV, creates the derived variables, drops the useless columns and produces `nhanes_2013_clean.csv` and a codebook to review by hand.

## Part A: download, merge, label

In [1]:
#%pip install pyreadstat pandas numpy

In [2]:
import os, urllib.request
import numpy as np
import pandas as pd
import pyreadstat
from functools import reduce

BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/"
DATA_DIR = "nhanes_data"
os.makedirs(DATA_DIR, exist_ok=True)

# All files of the 2013-2014 cycle are joined on SEQN.
# GHB_H is used ONLY for the label (HbA1c) and must not end up among the features.
names = ["DEMO_H",                                   # demographics, sample weights
         "BMX_H", "BPX_H",                           # exam: body measures, blood pressure
         "DR1TOT_H",                                 # diet, day 1
         "PAQ_H", "SMQ_H", "SLQ_H", "ALQ_H",         # lifestyle: activity, smoking, sleep, alcohol
         "DBQ_H", "CBQ_H",                           # diet behavior / consumer behavior
         "HIQ_H", "HUQ_H",                           # health insurance, healthcare utilization
         "MCQ_H",                                    # medical conditions
         "GHB_H",                                    # laboratory: HbA1c (label)
         "DIQ_H"]                                    # diabetes questionnaire (label only)

The download happens only once: the files stay in `nhanes_data/` and are read from there on later runs. While reading, I also collect the variable descriptions (codebook).

In [3]:
def download(name):
    path = f"{DATA_DIR}/{name}.xpt"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE}{name}.xpt", path)
    return path

dfs, rows = {}, []
for n in names:
    d, meta = pyreadstat.read_xport(download(n))
    dfs[n] = d
    rows += [(c, meta.column_names_to_labels.get(c, ""), n) for c in d.columns]
    print(f"{n:<10} {d.shape[0]:>6} rows  {d.shape[1]:>4} variables")

codebook = pd.DataFrame(rows, columns=["var", "label", "file"]).drop_duplicates("var")
codebook.to_csv("codebook.csv", index=False)

DEMO_H      10175 rows    47 variables
BMX_H        9813 rows    26 variables
BPX_H        9813 rows    23 variables
DR1TOT_H     9813 rows   168 variables
PAQ_H        9484 rows    96 variables
SMQ_H        7168 rows    32 variables
SLQ_H        6464 rows     4 variables
ALQ_H        5924 rows    10 variables
DBQ_H       10175 rows    51 variables
CBQ_H       10175 rows     6 variables
HIQ_H       10175 rows    17 variables
HUQ_H       10175 rows    10 variables
MCQ_H        9770 rows    95 variables
GHB_H        6979 rows     2 variables
DIQ_H        9770 rows    54 variables


### Merge and analysis sample

The files do not have the same rows because each part targets a different group (all interviewed people, only those examined, only adults...). We start from the demographics file, left join on `SEQN`, and then define the sample.

In [4]:
df = reduce(lambda l, r: l.merge(r, on="SEQN", how="left"),
            [dfs["DEMO_H"]] + [d for n, d in dfs.items() if n != "DEMO_H"])
print("Interviewed:", len(df), "| columns:", df.shape[1])

# check: columns with the same name in different files (they would end with _x / _y)
dup = [c for c in df.columns if c.endswith(("_x", "_y"))]
print("Columns duplicated across files:", dup if dup else "none")

df = df[(df.RIDAGEYR >= 20)        # adults
        & (df.RIDSTATR == 2)       # exam at the mobile center completed
        & df.LBXGH.notna()]        # HbA1c available (needed for the label)
df = df[df.RIDEXPRG != 1]          # exclude pregnancies (NaN = not applicable, kept)
print("Analysis sample:", len(df))

Interviewed: 10175 | columns: 627
Columns duplicated across files: none
Analysis sample: 5331


### 3-class label (HbA1c + diagnosis)

0 = low risk (HbA1c < 5.7), 1 = prediabetes (5.7 - 6.4, or "borderline" answer), 2 = diabetes (>= 6.5, or diagnosed by a doctor).
`LBXGH` and `DIQ010` are only used to build the label (it will be excluded from the features).

In [5]:
# DIQ010: doctor told you have diabetes (1 = yes, 2 = no, 3 = borderline, 7/9 = refused/don't know)
lab_diabetes = df.LBXGH >= 6.5
lab_prediab  = (df.LBXGH >= 5.7) & (df.LBXGH < 6.5)

df["risk"] = np.select(
    [lab_diabetes | (df.DIQ010 == 1),          # class 2: lab OR known diagnosis
     lab_prediab  | (df.DIQ010 == 3)],         # class 1: lab OR "borderline"
    [2, 1], default=0)

print(df["risk"].value_counts(normalize=True).sort_index().round(3))
print(pd.crosstab(df.DIQ010, df.risk))        # quick check of how the two sources agree

df.to_csv("nhanes_2013_merged.csv", index=False)


risk
0    0.602
1    0.243
2    0.155
Name: proportion, dtype: float64
risk       0     1    2
DIQ010                 
1.0        0     0  680
2.0     3207  1170  109
3.0        0   123   39
9.0        3     0    0


---
## Part B: cleaning and selection

From here on we always restart from the CSV, with no re-download.

In [6]:
import numpy as np
import pandas as pd

df = pd.read_csv("nhanes_2013_merged.csv")
codebook = pd.read_csv("codebook.csv")

# Missing-value percentage computed BEFORE creating derived variables or dropping anything
miss = df.isna().mean() * 100
to_drop = list(miss[miss > 50].index)

print(df.shape[0], "rows,", df.shape[1], "columns")
print(pd.cut(miss, [-0.1, 5, 20, 50, 100],
             labels=["<5%", "5-20%", "20-50%", ">50%"]).value_counts())

5331 rows, 628 columns
>50%      354
<5%       129
5-20%     108
20-50%     37
Name: count, dtype: int64


### Derived variables

In NHANES many NaNs are not missing data but questions that were never asked (someone who does not smoke does not answer how many cigarettes they smoke). Before dropping the columns with more than 50% NaN, we build variables that handle these cases correctly. The original columns are still needed, so this cell must run before the drops.

Variable names and codes (7/9, 77/99, 7777/9999 = refused or don't know) should be checked against the `... Doc` documentation of each file.

In [7]:
# --- Smoking: 0 = never, 1 = former smoker, 2 = current smoker ---
df.loc[df.SMD650 >= 777, "SMD650"] = np.nan                    # refused / don't know
df["smoking_status"] = np.select(
    [df.SMQ020 == 2, df.SMQ040.isin([1, 2]), df.SMQ040 == 3],
    [0, 2, 1], default=np.nan)
df["cigarettes_per_day"] = (df.SMD650.where(df.smoking_status == 2, 0)
                              .where(df.smoking_status.notna()))

# --- Physical activity: minutes per week ---
for c in ["PAD615", "PAD630", "PAD645", "PAD660", "PAD675"]:
    df.loc[df[c] >= 7777, c] = np.nan
for c in ["PAQ610", "PAQ625", "PAQ640", "PAQ655", "PAQ670"]:
    df.loc[df[c] >= 77, c] = np.nan

def weekly_minutes(flag, days, minutes):
    m = (df[days] * df[minutes]).where(df[flag] == 1, 0)       # answer "no" -> 0 minutes
    return m.where(df[flag].isin([1, 2]))                      # refused / missing -> NaN

df["vigorous_min"] = (weekly_minutes("PAQ605", "PAQ610", "PAD615")
                      + weekly_minutes("PAQ650", "PAQ655", "PAD660"))
df["moderate_min"] = (weekly_minutes("PAQ620", "PAQ625", "PAD630")
                      + weekly_minutes("PAQ635", "PAQ640", "PAD645")
                      + weekly_minutes("PAQ665", "PAQ670", "PAD675"))
df["equivalent_min"] = df.moderate_min + 2 * df.vigorous_min   # WHO convention

# --- Insurance (NaN = box not ticked) and hospital stays ---
df["medicare"] = df.HIQ031B.notna().astype(int)
df["medicaid"] = df.HIQ031D.notna().astype(int)
df.loc[df.HUD080 >= 77777, "HUD080"] = np.nan                  # check the code in the documentation
df["hospital_stays"] = df.HUD080.where(df.HUQ071 == 1, 0)

# --- Blood pressure: mean of the repeated readings ---
df["sbp_mean"] = df[["BPXSY1", "BPXSY2", "BPXSY3"]].mean(axis=1)
df["dbp_mean"] = df[["BPXDI1", "BPXDI2", "BPXDI3"]].mean(axis=1)

derived = {
    "smoking_status": "Smoking: 0 never, 1 former, 2 current",
    "cigarettes_per_day": "Cigarettes per day (0 for non-smokers)",
    "vigorous_min": "Minutes/week of vigorous activity",
    "moderate_min": "Minutes/week of moderate activity",
    "equivalent_min": "Moderate-equivalent minutes (moderate + 2 x vigorous)",
    "medicare": "Covered by Medicare (0/1)",
    "medicaid": "Covered by Medicaid (0/1)",
    "hospital_stays": "Overnight hospital stays in the last year (0 if none)",
    "sbp_mean": "Mean systolic blood pressure",
    "dbp_mean": "Mean diastolic blood pressure",
}
df[list(derived)].describe().T.round(1)

/var/folders/1z/cxmn5q45049_ns0thyhh438w0000gp/T/ipykernel_27052/1153332154.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["smoking_status"] = np.select(
/var/folders/1z/cxmn5q45049_ns0thyhh438w0000gp/T/ipykernel_27052/1153332154.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["cigarettes_per_day"] = (df.SMD650.where(df.smoking_status == 2, 0)
/var/folders/1z/cxmn5q45049_ns0thyhh438w0000gp/T/ipykernel_27052/1153332154.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling

,count,mean,std,min,25%,50%,75%,max
smoking_status,5329.0,0.6,0.8,0.0,0.0,0.0,1.0,2.0
cigarettes_per_day,5316.0,2.2,5.7,0.0,0.0,0.0,0.0,90.0
vigorous_min,5324.0,191.8,494.7,0.0,0.0,0.0,120.0,6480.0
moderate_min,5317.0,402.4,694.1,0.0,0.0,125.0,450.0,7200.0
equivalent_min,5316.0,785.9,1380.8,0.0,0.0,240.0,840.0,14760.0
medicare,5331.0,0.2,0.4,0.0,0.0,0.0,0.0,1.0
medicaid,5331.0,0.1,0.3,0.0,0.0,0.0,0.0,1.0
hospital_stays,5331.0,0.2,0.6,0.0,0.0,0.0,0.0,6.0
sbp_mean,5154.0,123.4,17.8,64.7,111.3,120.0,133.3,228.7
dbp_mean,5154.0,69.6,12.7,0.0,63.3,70.7,77.0,116.7


### Dropping columns

We drop: columns with more than 50% missing values, columns always excluded by choice (leakage or filter already applied), the original columns already transformed, and the near-constant ones (same value for more than 99% of people).

In [8]:
always_drop = ["DRQSDT7",              # diabetic diet: reveals the diagnosis (leakage)
               "DMDYRSUS", "DMQADFC",  # not relevant
               "RIDEXPRG"]             # pregnancy: already used as a filter

used = ["SMQ020", "SMQ040", "SMD650", "HIQ031B", "HIQ031D", "HUD080", "HUQ071",
        "PAQ605", "PAQ610", "PAD615", "PAQ620", "PAQ625", "PAD630", "PAQ635",
        "PAQ640", "PAD645", "PAQ650", "PAQ655", "PAD660", "PAQ665", "PAQ670", "PAD675",
        "BPXSY1", "BPXSY2", "BPXSY3", "BPXDI1", "BPXDI2", "BPXDI3"]

drop_cols = list(dict.fromkeys(to_drop + always_drop + used))
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
label_cols = ["LBXGH", "DIQ010", "risk"]
df = df.drop(columns=[c for c in df.columns if c.startswith("DIQ") and c not in label_cols])
print("After the drops:", df.shape)

# near-constant columns (keys, label columns and weights are protected)
protected = ["SEQN", "risk", "LBXGH", "DIQ010", "WTMEC2YR", "WTINT2YR", "SDMVPSU", "SDMVSTRA"]
def near_constant(s):
    vc = s.value_counts(normalize=True)
    return len(vc) <= 1 or vc.iloc[0] > 0.99

near_const = [c for c in df.columns if c not in protected and near_constant(df[c])]
print("Near-constant columns dropped:", near_const)
df = df.drop(columns=near_const)
print("Final shape:", df.shape)

After the drops: (5331, 266)
Near-constant columns dropped: ['SDDSRVYR', 'RIDSTATR', 'SIAPROXY', 'FIAPROXY', 'MIAPROXY', 'BPAARM', 'BPXPTY', 'DRABF', 'DR1MNRSP', 'DRD370J', 'DRD370L', 'DRD370P', 'DRD370S', 'DRD370V', 'PAAQUEX', 'SMAQUEX2', 'HIQ031A', 'MCQ082']
Final shape: (5331, 248)


### Columns NOT to use as features

IDs, status flags, sample weights, and the label itself (`LBXGH`, `risk`). They stay in the file for weighted analyses, but must not be fed to the model.

In [9]:
non_feature = ["SEQN", "SDDSRVYR", "RIDSTATR", "RIDEXMON", "BMDSTATS",
               "SDMVPSU", "SDMVSTRA", "WTMEC2YR", "WTINT2YR",
               "WTDRD1", "WTDR2D", "DR1DRSTZ", "DR1EXMER", "SMAQUEX2",
               "LBXGH", "DIQ010", "risk"]
non_feature = [c for c in non_feature if c in df.columns]

feature_candidates = [c for c in df.columns if c not in non_feature]
print(len(feature_candidates), "candidate features")

234 candidate features


### Codebook of the remaining columns, to review one by one

The file `codebook_remaining.csv` has a `keep` column to fill in by hand (yes / no / derive). The non-features are already marked "no". Watch out for redundant pairs, for example `RIDRETH1` and `RIDRETH3` (two codings of the same ethnicity variable): only one is needed.

In [10]:
remaining = codebook[codebook["var"].isin(df.columns)].copy()
new_rows = pd.DataFrame({"var": list(derived), "label": list(derived.values()), "file": "derived"})
remaining = pd.concat([remaining, new_rows], ignore_index=True)
remaining = remaining[remaining["var"].isin(df.columns)]

remaining["missing_%"] = remaining["var"].map(df.isna().mean() * 100).round(1)
remaining["keep"] = np.where(remaining["var"].isin(non_feature), "no", "")
remaining = remaining.sort_values(["file", "missing_%"])
remaining.to_csv("codebook_remaining.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 80)
for f, g in remaining.groupby("file"):
    print(f"\n===== {f} ({len(g)} variables) =====")
    print(g[["var", "label", "missing_%"]].to_string(index=False))


===== ALQ_H (6 variables) =====
    var                                    label  missing_%
 ALQ101     Had at least 12 alcohol drinks/1 yr?        8.1
ALQ120Q How often drink alcohol over past 12 mos       22.0
 ALQ151  Ever have 4/5 or more drinks every day?       22.0
ALQ120U      # days drink alcohol per wk, mo, yr       37.5
 ALQ130 Avg # alcoholic drinks/day - past 12 mos       37.5
ALQ141Q     # days have 4/5 drinks - past 12 mos       37.5

===== BMX_H (11 variables) =====
     var                                    label  missing_%
BMDSTATS      Body Measures Component Status Code        0.0
   BMXWT                              Weight (kg)        0.9
   BMXHT                     Standing Height (cm)        1.0
  BMXBMI                Body Mass Index (kg/m**2)        1.1
 BMXARMC                   Arm Circumference (cm)        4.0
 BMXARML                    Upper Arm Length (cm)        4.1
BMXWAIST                 Waist Circumference (cm)        5.0
  BMXLEG                 

In [ ]:
print(len(df))
df.to_csv("nhanes_2013_clean.csv", index=False)